# タイヤグリップ分析：合計G vs タイヤ空気圧・温度

このノートブックでは、合計加速度（横Gと縦Gの合成）とタイヤ空気圧または温度の関係を各コーナーごとに分析します。

## 分析内容

- **2x2 グリップエンベロープ図**: 各コーナー（FL、FR、RL、RR）の合計Gパーセンタイル vs タイヤ空気圧/温度
- **統計テーブル**: 各コーナーの平均値、標準偏差

## 分析結果の読み方

| パターン | 意味 |
|---------|--------|
| 上昇するエンベロープ | タイヤ指標が高いほどグリップが大きい |
| 下降するエンベロープ | タイヤ指標が高いほどグリップが小さい |
| 平坦なエンベロープ | この範囲ではタイヤ指標がグリップにほとんど影響しない |
| フロント/リアの差 | タイヤ空気圧バランスの問題の可能性 |
| 左右の差 | タイヤ摩耗や空気圧の偏りの可能性 |

## 自分のデータを使う

1. **最初のセルを実行**してパッケージをインストールし、アップロードウィジェットを表示
2. **「Choose File」をクリック**して `.xrk` または `.xrz` ファイルを選択
3. TPMSチャンネル名が異なる場合は**チャンネル名を設定**
4. **残りのセルをすべて実行**してデータを分析

## 必要なチャンネル

- 横G・縦Gチャンネル（例：`LateralAcc`、`InlineAcc`）
- TPMS空気圧または温度チャンネル（例：`TPMS_Press_LF`、`TPMS_Temp_LF`）

**注：** このノートブックはJupyterLite（ブラウザ）と通常のJupyterLabの両方で動作します。

In [ ]:
# 必要なパッケージをインストール（JupyterLite用、通常のJupyterLabではインストール済みならスキップ）
%pip install -q pandas plotly libxrk motorsports-data-notebook jinja2 ipywidgets

# Rustパーサーバックエンドを使用して読み込みを高速化
import os

os.environ["LIBXRK_BACKEND"] = "rust"

# ヘルパー関数のインポート
from motorsports_data_notebook.channels import get_top_laps
from motorsports_data_notebook.tire_grip import (
    analyze_tire_grip_multi_lap,
    format_tire_grip_stats_table,
)
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_tire_grip_scatter,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker

# セッションピッカーとチャンネル設定
# 自分のファイルをアップロード — ベストの103%以内の全ラップが自動的に分析されます
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    show_lap_picker=False,
    channel_mapping={
        "lateral_g": "LateralAcc",
        "inline_g": "InlineAcc",
        "tpms_press_fl": "TPMS_Press_LF",
        "tpms_press_fr": "TPMS_Press_RF",
        "tpms_press_rl": "TPMS_Press_LR",
        "tpms_press_rr": "TPMS_Press_RR",
        "tpms_temp_fl": "TPMS_Temp_LF",
        "tpms_temp_fr": "TPMS_Temp_RF",
        "tpms_temp_rl": "TPMS_Temp_LR",
        "tpms_temp_rr": "TPMS_Temp_RR",
    },
)
session.display()

In [ ]:
# ラップデータをpandas DataFrameとして取得
laps = session.get_laps()

In [ ]:
# ラップタイムテーブルを表示
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

## 設定

タイヤの**空気圧**と**温度**のどちらを分析するかを、下の `metric_mode` で選択してください。

In [ ]:
# metric_modeを"temperature"に変更すると、空気圧の代わりにタイヤ温度をプロット
metric_mode = "pressure"  # "pressure"（空気圧）または "temperature"（温度）

# バケットごとの合計Gのパーセンタイル（例：99.9 = グリップエンベロープ）
percentile = 99.9

## タイヤグリップ vs 空気圧/温度

各サブプロットは、1つのコーナーのグリップエンベロープを表示します — 合計G（横G + 縦Gの合成加速度）のバケット化パーセンタイル（デフォルトP99.9）と選択したタイヤ指標の関係です。各空気圧/温度値で利用可能な最大グリップが明らかになります。

In [ ]:
# 上位ラップ全体で分析を実行（ベストラップタイムの103%以内）
log = session.get_log()
channel_names = session.get_channel_names()

top_laps = get_top_laps(laps, threshold_pct=1.03)
lap_numbers = top_laps["num"].astype(int).tolist()

print(f"ベストラップタイム: {format_lap_time(laps['lap_time'].min())}")
print(f"ベストの103%以内の{len(top_laps)}ラップを使用して分析")

result = analyze_tire_grip_multi_lap(
    log, lap_numbers, channel_names, metric_mode=metric_mode, percentile=percentile
)

# グリップエンベロープをプロット
mode_label = "空気圧" if metric_mode == "pressure" else "温度"
fig = plot_tire_grip_scatter(
    result,
    title=f"タイヤグリップ（{mode_label}） - 上位{len(top_laps)}ラップ (P{percentile})",
)
show_fig(fig)

## 統計

各コーナーの合計Gと選択したタイヤ指標の平均値、標準偏差のサマリーです。

In [ ]:
# 統計テーブル
stats_df = format_tire_grip_stats_table(result)
# 全数値列を小数点以下2桁でフォーマット
float_cols = {col: "{:.2f}" for col in stats_df.columns if col != "Corner"}
stats_df.style.format(float_cols)  # type: ignore[arg-type]